In [ ]:
# Cell 1 — mount, copy tar local, extract to scratch, verify
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, time
TAR_DRIVE = '/content/drive/MyDrive/BraTS/BraTS2026/resenc_preprocessed_COMPLETE.tar'
TAR_LOCAL = '/content/resenc_preprocessed_COMPLETE.tar'
SCRATCH   = '/mnt/local-scratch'
PRE       = f'{SCRATCH}/nnUNet_preprocessed'

# copy off Drive (sequential read, ~25-35 min on a slow Drive day)
t0 = time.time()
print("copying tar to local disk...")
!cp "{TAR_DRIVE}" "{TAR_LOCAL}"
print(f"  copied in {(time.time()-t0)/60:.1f} min")

# extract to scratch (fast NVMe)
os.makedirs(PRE, exist_ok=True)
t1 = time.time()
print("extracting to scratch...")
!cd "{PRE}" && tar xf "{TAR_LOCAL}"
print(f"  extracted in {(time.time()-t1)/60:.1f} min")

# verify
ds   = f'{PRE}/Dataset501_BraTSMET2026'
tree = f'{ds}/nnUNetPlans_3d_fullres'
print("\n3d_fullres files:", len(os.listdir(tree)), "(expect 3888)")
print("gt_segmentations:", len(os.listdir(f'{ds}/gt_segmentations')), "(expect 1296)")
print("ResEnc plan:", os.path.exists(f'{ds}/nnUNetResEncUNetLPlans.json'))
print("splits:", os.path.exists(f'{ds}/splits_final.json'))

Mounted at /content/drive
copying tar to local disk...
  copied in 35.5 min
extracting to scratch...
  extracted in 3.7 min

3d_fullres files: 3888 (expect 3888)
gt_segmentations: 1296 (expect 1296)
ResEnc plan: True
splits: True


In [ ]:
import numpy as np
print("numpy before:", np.__version__)
!pip install nnunetv2==2.7.0 --quiet 2>&1 | tail -3
print("\n⚠️  RESTART RUNTIME, then run Cell 3.")

numpy before: 2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.

⚠️  RESTART RUNTIME, then run Cell 3.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PRE = '/mnt/local-scratch/nnUNet_preprocessed'
os.environ['nnUNet_raw']          = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = PRE
os.environ['nnUNet_results']      = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_results'
os.environ['nnUNet_compile']      = 'F'

import numpy as np
tree = f'{PRE}/Dataset501_BraTSMET2026/nnUNetPlans_3d_fullres'
print("numpy:", np.__version__, "| compile:", os.environ['nnUNet_compile'])
print("scratch files:", len(os.listdir(tree)), "(expect 3888)")

# launch fold 3 fresh (no --c, this is a new fold)
!nnUNetv2_train 501 3d_fullres 3 \
  -p nnUNetResEncUNetLPlans \
  -tr nnUNetTrainer_500epochs \
  --npz

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
numpy: 2.5.0 | compile: F
scratch files: 3888 (expect 3888)
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-06-27 05:46:03.502131: do_dummy_2d_data_aug: False
2026-06-27 05:46:03.508752: Using splits from existing split file: /mnt/local-scratch/nnUNet_preprocessed/Dataset501_BraTSMET2026/splits_final.json
2026-06-27 05:46:03.512213: The split file contains 5 splits.
2026-06-27 05:46:03.541080: Desired fold for training: 3
2026-06-27 05:46:03.546014: This split has 1037 training and 259

In [ ]:
# Cell 1 — mount, confirm fold 3 checkpoint survived, then stage tar
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, time

# first: confirm there's a checkpoint to resume from (don't stage 35 min for nothing)
RES = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_results/Dataset501_BraTSMET2026/nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres/fold_3'
print("fold_3 checkpoints on Drive:")
for ck in ['checkpoint_latest.pth','checkpoint_best.pth']:
    p = f'{RES}/{ck}'
    print(f"  {ck}:", "✓ "+str(round(os.path.getsize(p)/1024**3,2))+" GB" if os.path.exists(p) else "✗ MISSING")

# stage tar to scratch
TAR_DRIVE = '/content/drive/MyDrive/BraTS/BraTS2026/resenc_preprocessed_COMPLETE.tar'
TAR_LOCAL = '/content/resenc_preprocessed_COMPLETE.tar'
PRE = '/mnt/local-scratch/nnUNet_preprocessed'
os.makedirs(PRE, exist_ok=True)

t0 = time.time()
print("\ncopying tar to local (~35 min)...")
!cp "{TAR_DRIVE}" "{TAR_LOCAL}"
print(f"  copied in {(time.time()-t0)/60:.1f} min")

t1 = time.time()
print("extracting to scratch...")
!cd "{PRE}" && tar xf "{TAR_LOCAL}"
print(f"  extracted in {(time.time()-t1)/60:.1f} min")

tree = f'{PRE}/Dataset501_BraTSMET2026/nnUNetPlans_3d_fullres'
print("\n3d_fullres files:", len(os.listdir(tree)), "(expect 3888)")

Mounted at /content/drive
fold_3 checkpoints on Drive:
  checkpoint_latest.pth: ✓ 0.76 GB
  checkpoint_best.pth: ✓ 0.76 GB

copying tar to local (~35 min)...
  copied in 35.8 min
extracting to scratch...
  extracted in 3.8 min

3d_fullres files: 3888 (expect 3888)


In [ ]:
import numpy as np
print("numpy before:", np.__version__)
!pip install nnunetv2==2.7.0 --quiet 2>&1 | tail -3
print("\n⚠️  RESTART RUNTIME now, then run Cell 3.")

numpy before: 2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.

⚠️  RESTART RUNTIME now, then run Cell 3.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PRE = '/mnt/local-scratch/nnUNet_preprocessed'
os.environ['nnUNet_raw']          = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = PRE
os.environ['nnUNet_results']      = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_results'
os.environ['nnUNet_compile']      = 'F'

import numpy as np
tree = f'{PRE}/Dataset501_BraTSMET2026/nnUNetPlans_3d_fullres'
print("numpy:", np.__version__, "| compile:", os.environ['nnUNet_compile'])
print("scratch files:", len(os.listdir(tree)), "(expect 3888)")

# resume fold 3 from checkpoint_latest (--c continues from ~ep350)
!nnUNetv2_train 501 3d_fullres 3 \
  -p nnUNetResEncUNetLPlans \
  -tr nnUNetTrainer_500epochs \
  --npz --c

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
numpy: 2.5.0 | compile: F
scratch files: 3888 (expect 3888)
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-06-28 16:12:14.254544: do_dummy_2d_data_aug: False
2026-06-28 16:12:14.263488: Using splits from existing split file: /mnt/local-scratch/nnUNet_preprocessed/Dataset501_BraTSMET2026/splits_final.json
2026-06-28 16:12:14.268270: The split file contains 5 splits.
2026-06-28 16:12:14.271857: Desired fold for training: 3
2026-06-28 16:12:14.274839: This split has 1037 training and 259

In [ ]:
# Re-mount + verify all four fold checkpoints survived
from google.colab import drive
drive.mount('/content/drive')

import os
RES = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_results/Dataset501_BraTSMET2026/nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres'
for f in ['fold_0','fold_1','fold_2','fold_3']:
    p = f'{RES}/{f}/checkpoint_best.pth'
    print(f"{f}: {'✓' if os.path.exists(p) else '✗ MISSING'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fold_0: ✓
fold_1: ✓
fold_2: ✓
fold_3: ✓


In [ ]:
import os, time
os.environ['nnUNet_compile'] = 'F'
INPUT = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw/Dataset501_BraTSMET2026/imagesTs'
OUT_F3 = '/content/fold3_pred'
os.makedirs(OUT_F3, exist_ok=True)
t0 = time.time()
!nnUNetv2_predict -i "{INPUT}" -o "{OUT_F3}" -d 501 -c 3d_fullres \
  -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_500epochs \
  -f 3 -chk checkpoint_best.pth --save_probabilities
print(f"\nfold 3 done in {(time.time()-t0)/60:.1f} min")


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 179 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 179 cases that I would like to predict

Predicting BraTS-MET-00833-000:
perform_everything_on_device: True
100% 1/1 [00:17<00:00, 17.03s/it]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00833-000

Predicting BraTS-MET-00834-000:
perform_everything_on_device: True
100% 1/1 [00:00<00:00,  1.61it/s]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00834-000

Predicting BraTS-MET-00835

In [ ]:
import os, numpy as np, nibabel as nib
PRED = '/content/fold3_pred'
niis = sorted([f for f in os.listdir(PRED) if f.endswith('.nii.gz')])
print(f"nii.gz files: {len(niis)} (expect 179)")

empty, et_present, label_counts = 0, 0, {}
for f in niis:
    arr = np.asanyarray(nib.load(f'{PRED}/{f}').dataobj)
    labs = np.unique(arr).astype(int).tolist()
    if labs == [0]: empty += 1
    if 3 in labs: et_present += 1
    for l in labs: label_counts[l] = label_counts.get(l, 0) + 1
print(f"empty: {empty}")
print(f"ET present: {et_present}/{len(niis)}")
print(f"label presence: {dict(sorted(label_counts.items()))}")
print("  (0=bg,1=NETC,2=SNFH,3=ET,4=RC)")

nii.gz files: 179 (expect 179)
empty: 11
ET present: 166/179
label presence: {0: 179, 1: 80, 2: 155, 3: 166, 4: 71}
  (0=bg,1=NETC,2=SNFH,3=ET,4=RC)


In [1]:
import os
print("fold3_pred exists:", os.path.exists('/content/fold3_pred'))
if os.path.exists('/content/fold3_pred'):
    print("files:", len(os.listdir('/content/fold3_pred')))

fold3_pred exists: False


In [2]:
import os
# check if nnunet survived (it won't if truly fresh runtime)
result = os.system('which nnUNetv2_predict')
print("nnUNet CLI available:", result == 0)
print("Drive mounted:", os.path.exists('/content/drive/MyDrive'))

nnUNet CLI available: False
Drive mounted: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np
print("numpy before:", np.__version__)
!pip install nnunetv2==2.7.0 --quiet 2>&1 | tail -3
print("\n⚠️  RESTART RUNTIME now, then run Cell 2.")

Mounted at /content/drive
numpy before: 2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.

⚠️  RESTART RUNTIME now, then run Cell 2.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
os.environ['nnUNet_raw']     = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw'
os.environ['nnUNet_results'] = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_results'
os.environ['nnUNet_compile'] = 'F'

INPUT  = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw/Dataset501_BraTSMET2026/imagesTs'
# output to DRIVE this time — survives disconnects
OUT_F3 = '/content/drive/MyDrive/BraTS/BraTS2026/predictions/fold3_pred'
os.makedirs(OUT_F3, exist_ok=True)

import numpy as np
print("numpy:", np.__version__, "| compile:", os.environ['nnUNet_compile'])

t0 = time.time()
!nnUNetv2_predict -i "{INPUT}" -o "{OUT_F3}" -d 501 -c 3d_fullres \
  -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_500epochs \
  -f 3 -chk checkpoint_best.pth --save_probabilities
print(f"\nfold 3 done in {(time.time()-t0)/60:.1f} min")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
numpy: 2.5.0 | compile: F
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 179 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 179 cases that I would like to predict
nnUNet_preprocessed is not defined and nnU-Net can not be used for prepr

In [3]:
import os, time
os.environ['nnUNet_compile'] = 'F'
INPUT  = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw/Dataset501_BraTSMET2026/imagesTs'
OUT_F0 = '/content/drive/MyDrive/BraTS/BraTS2026/predictions/fold0_pred'
os.makedirs(OUT_F0, exist_ok=True)
t0 = time.time()
!nnUNetv2_predict -i "{INPUT}" -o "{OUT_F0}" -d 501 -c 3d_fullres \
  -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_500epochs \
  -f 0 -chk checkpoint_best.pth --save_probabilities
print(f"\nfold 0 done in {(time.time()-t0)/60:.1f} min")

nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 179 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 179 cases that I would like to predict
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_preprocessed is 

In [4]:
import os, time
os.environ['nnUNet_compile'] = 'F'
INPUT  = '/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw/Dataset501_BraTSMET2026/imagesTs'
OUT_F1 = '/content/drive/MyDrive/BraTS/BraTS2026/predictions/fold1_pred'
os.makedirs(OUT_F1, exist_ok=True)
t0 = time.time()
!nnUNetv2_predict -i "{INPUT}" -o "{OUT_F1}" -d 501 -c 3d_fullres \
  -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_500epochs \
  -f 1 -chk checkpoint_best.pth --save_probabilities
print(f"\nfold 1 done in {(time.time()-t0)/60:.1f} min")

nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 179 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 179 cases that I would like to predict
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_preprocessed is 

In [5]:
import os
P = '/content/drive/MyDrive/BraTS/BraTS2026/predictions'
for f in ['fold0_pred','fold1_pred','fold3_pred']:
    path = f'{P}/{f}'
    n_nii = len([x for x in os.listdir(path) if x.endswith('.nii.gz')]) if os.path.exists(path) else 0
    n_npz = len([x for x in os.listdir(path) if x.endswith('.npz')]) if os.path.exists(path) else 0
    print(f"{f}: {n_nii} nii.gz, {n_npz} npz")

fold0_pred: 179 nii.gz, 179 npz
fold1_pred: 179 nii.gz, 179 npz
fold3_pred: 179 nii.gz, 179 npz


In [6]:
# Cell 5 — ensemble folds 0+1+3 (all RC-alive)
import os, time
P = '/content/drive/MyDrive/BraTS/BraTS2026/predictions'
ENS = f'{P}/ensemble013_pred'
os.makedirs(ENS, exist_ok=True)
t0 = time.time()
!nnUNetv2_ensemble \
  -i "{P}/fold0_pred" "{P}/fold1_pred" "{P}/fold3_pred" \
  -o "{ENS}" -np 4
print(f"\nensemble 0+1+3 done in {(time.time()-t0)/60:.1f} min")


ensemble 0+1+3 done in 12.5 min


In [7]:
import os, numpy as np, nibabel as nib
PRED = '/content/drive/MyDrive/BraTS/BraTS2026/predictions/ensemble013_pred'
niis = sorted([f for f in os.listdir(PRED) if f.endswith('.nii.gz')])
print(f"nii.gz files: {len(niis)} (expect 179)")

empty, et_present, label_counts = 0, 0, {}
for f in niis:
    arr = np.asanyarray(nib.load(f'{PRED}/{f}').dataobj)
    labs = np.unique(arr).astype(int).tolist()
    if labs == [0]: empty += 1
    if 3 in labs: et_present += 1
    for l in labs: label_counts[l] = label_counts.get(l, 0) + 1
print(f"empty: {empty}")
print(f"ET present: {et_present}/{len(niis)}")
print(f"label presence: {dict(sorted(label_counts.items()))}")
print("  (0=bg,1=NETC,2=SNFH,3=ET,4=RC)")

nii.gz files: 179 (expect 179)
empty: 15
ET present: 163/179
label presence: {0: 179, 1: 82, 2: 152, 3: 163, 4: 65}
  (0=bg,1=NETC,2=SNFH,3=ET,4=RC)


In [8]:
import os, zipfile, shutil
PRED = '/content/drive/MyDrive/BraTS/BraTS2026/predictions/ensemble013_pred'
ZIP  = '/content/drive/MyDrive/BraTS/BraTS2026/BraTS_Uniandes_ensemble_f0f1f3.zip'
niis = sorted([f for f in os.listdir(PRED) if f.endswith('.nii.gz')])
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in niis: z.write(f'{PRED}/{f}', arcname=f)
with zipfile.ZipFile(ZIP) as z: names = z.namelist()
print(f"zipped {len(names)} files (expect 179)")
print("all flat:", all('/' not in n for n in names), "| all nii.gz:", all(n.endswith('.nii.gz') for n in names))
from google.colab import files
files.download(ZIP)
print("saved to Drive + downloading")

zipped 179 files (expect 179)
all flat: True | all nii.gz: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

saved to Drive + downloading


In [9]:
import os, zipfile
PRED = '/content/drive/MyDrive/BraTS/BraTS2026/predictions/fold3_pred'
ZIP  = '/content/drive/MyDrive/BraTS/BraTS2026/BraTS_Uniandes_fold3_solo.zip'

# confirm fold3_pred still has its files
niis = sorted([f for f in os.listdir(PRED) if f.endswith('.nii.gz')])
print(f"fold3_pred nii.gz: {len(niis)} (expect 179)")

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in niis: z.write(f'{PRED}/{f}', arcname=f)
with zipfile.ZipFile(ZIP) as z: names = z.namelist()
print(f"zipped {len(names)} files (expect 179)")
print("all flat:", all('/' not in n for n in names), "| all nii.gz:", all(n.endswith('.nii.gz') for n in names))
from google.colab import files
files.download(ZIP)
print("saved to Drive + downloading")

fold3_pred nii.gz: 179 (expect 179)
zipped 179 files (expect 179)
all flat: True | all nii.gz: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

saved to Drive + downloading
